In [1]:
import time
import networkx as nx, pandas as pd, numpy as np
from networkx.algorithms import community as nxc

G = nx.read_edgelist("../data/facebook_combined.txt.gz", nodetype=int)
G.remove_edges_from(nx.selfloop_edges(G))

# Louvain
t = time.time()
louv = nxc.louvain_communities(G, seed=42)
t_louv = time.time() - t
mod_louv = nxc.modularity(G, louv)

# Label Propagation: chạy 10 lần vì thuật toán có tính ngẫu nhiên
runs = []
for s in range(10):
    t = time.time()
    lp = list(nxc.asyn_lpa_communities(G, seed=s))
    runs.append((time.time() - t, nxc.modularity(G, lp), len(lp), lp))
best = max(runs, key=lambda r: r[1])

so_sanh = pd.DataFrame({
    "Tiêu chí": ["Số cộng đồng", "Modularity", "Cộng đồng lớn nhất", "Thời gian (giây)"],
    "Louvain": [len(louv), round(mod_louv, 4), max(len(c) for c in louv), round(t_louv, 2)],
    "Label Propagation (tốt nhất / 10 lần)": [best[2], round(best[1], 4), max(len(c) for c in best[3]), round(best[0], 2)],
})
print(so_sanh.to_string(index=False))
print("LPA modularity TB ± độ lệch chuẩn:",
      round(np.mean([r[1] for r in runs]), 4), "±", round(np.std([r[1] for r in runs]), 4))
print("LPA số cộng đồng min-max:", min(r[2] for r in runs), "-", max(r[2] for r in runs))

          Tiêu chí  Louvain  Label Propagation (tốt nhất / 10 lần)
      Số cộng đồng  16.0000                                68.0000
        Modularity   0.8349                                 0.8172
Cộng đồng lớn nhất 548.0000                               484.0000
  Thời gian (giây)   2.0900                                 0.7400
LPA modularity TB ± độ lệch chuẩn: 0.8129 ± 0.0024
LPA số cộng đồng min-max: 66 - 75


In [2]:
# Lưu nhãn cộng đồng (Louvain) cho từng nút
louv_sorted = sorted(louv, key=len, reverse=True)
label = {n: i for i, c in enumerate(louv_sorted) for n in c}
nx.set_node_attributes(G, label, "community")
pd.Series(label, name="community").rename_axis("node").reset_index().to_csv("../data/louvain_labels.csv", index=False)

# Bảng 10 cộng đồng lớn nhất
rows = []
for i, c in enumerate(louv_sorted[:10]):
    sub = G.subgraph(c)
    rows.append({"Cộng đồng": i, "Số nút": len(c), "Số cạnh nội bộ": sub.number_of_edges(),
                 "Mật độ nội bộ": round(nx.density(sub), 3)})
bang = pd.DataFrame(rows)
print(bang.to_string(index=False))
bang.to_csv("../data/top_communities.csv", index=False)

# Xuất file cho Gephi
nx.write_gexf(G, "../data/facebook_louvain.gexf")

 Cộng đồng  Số nút  Số cạnh nội bộ  Mật độ nội bộ
         0     548            5356          0.036
         1     535            8691          0.061
         2     435           16687          0.177
         3     431            6060          0.065
         4     423           11422          0.128
         5     350            2845          0.047
         6     324            6303          0.120
         7     237           16543          0.592
         8     226            5600          0.220
         9     206            1983          0.094
